# 👥 Customer Segmentation with RFM Analysis & K-Means Clustering

**Author:** Allen Day  
**Goal:** Segment 52K customers into actionable groups using RFM features and K-Means clustering

---

## 0. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta

from rfm_utils import (
    compute_rfm, scale_rfm, rfm_summary,
    elbow_and_silhouette, plot_elbow_silhouette,
    fit_kmeans, profile_segments,
    plot_rfm_heatmap, plot_segment_distribution
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✅')

## 1. Load & Inspect Data

In [ ]:
# ── Simulate 52K customers / 180K transactions (24 months) ──
np.random.seed(42)
N_CUSTOMERS = 52_000
N_TRANSACTIONS = 180_000

customer_ids = np.random.choice(range(1, N_CUSTOMERS + 1), size=N_TRANSACTIONS)
start_date = datetime(2022, 1, 1)
dates = [start_date + timedelta(days=int(d)) for d in np.random.randint(0, 730, N_TRANSACTIONS)]
values = np.abs(np.random.lognormal(mean=4.8, sigma=1.1, size=N_TRANSACTIONS))
categories = np.random.choice(['Skincare', 'Supplements', 'Devices', 'Bundles'], N_TRANSACTIONS)

df = pd.DataFrame({
    'customer_id': customer_ids,
    'transaction_date': dates,
    'order_value': values.round(2),
    'product_category': categories
})

print(f'Dataset shape: {df.shape}')
print(f'Unique customers: {df.customer_id.nunique():,}')
print(f'Date range: {df.transaction_date.min().date()} → {df.transaction_date.max().date()}')
df.head()

In [ ]:
# Check for missing values
print('Missing values:')
print(df.isnull().sum())
print(f'\nOrder value range: ${df.order_value.min():.2f} – ${df.order_value.max():.2f}')
print(f'Avg order value: ${df.order_value.mean():.2f}')

## 2. RFM Feature Engineering

In [ ]:
rfm = compute_rfm(
    df,
    customer_col='customer_id',
    date_col='transaction_date',
    value_col='order_value'
)

print(f'RFM table shape: {rfm.shape}')
rfm.head(10)

In [ ]:
rfm_summary(rfm)

In [ ]:
# Distribution plots
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, color in zip(axes, ['Recency','Frequency','Monetary'],
                           ['#3498db','#2ecc71','#e74c3c']):
    axes_data = rfm[col].clip(upper=rfm[col].quantile(0.99))  # cap outliers
    ax.hist(axes_data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.set_title(f'{col} Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
plt.suptitle('RFM Feature Distributions (99th pct cap)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Data Normalization

In [ ]:
scaled, scaler = scale_rfm(rfm)
print(f'Scaled array shape: {scaled.shape}')
print(f'Mean (should ≈ 0): {scaled.mean(axis=0).round(3)}')
print(f'Std  (should ≈ 1): {scaled.std(axis=0).round(3)}')

## 4. Optimal K Selection

In [ ]:
scores = elbow_and_silhouette(scaled, k_range=range(2, 10))
plot_elbow_silhouette(scores)

**Finding:** Both the elbow curve and silhouette score point to **k=4** as optimal.

## 5. K-Means Clustering (k=4)

In [ ]:
rfm = fit_kmeans(rfm, scaled, k=4)
print(rfm['Segment'].value_counts())
rfm.head()

## 6. Segment Profiling

In [ ]:
profile = profile_segments(rfm)

In [ ]:
plot_segment_distribution(rfm)

In [ ]:
plot_rfm_heatmap(rfm)

## 7. Interactive 3D Scatter

In [ ]:
sample = rfm.sample(n=min(5000, len(rfm)), random_state=42)

fig = px.scatter_3d(
    sample, x='Recency', y='Frequency', z='Monetary',
    color='Segment',
    title='Customer Segments — 3D RFM View',
    labels={'Recency':'Days Since Last Purchase',
            'Frequency':'# Purchases',
            'Monetary':'Total Spend ($)'},
    opacity=0.6,
    height=600
)
fig.show()

## 8. Revenue Contribution by Segment

In [ ]:
rev_by_seg = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)
rev_pct = (rev_by_seg / rev_by_seg.sum() * 100).round(1)

print('Revenue share by segment:')
for seg, pct in rev_pct.items():
    print(f'  {seg}: {pct}%')

In [ ]:
fig = px.pie(
    values=rev_by_seg.values,
    names=rev_by_seg.index,
    title='Revenue Share by Customer Segment',
    hole=0.4
)
fig.show()

## 9. Business Recommendations

| Segment | Strategy |
|---------|----------|
| 💎 Champions | VIP loyalty program, exclusive early access, referral incentives |
| 🚀 Promising | Upsell bundles, loyalty tier upgrades, personalized product recs |
| 🌱 At-Risk | Win-back email flow, time-limited discount (15–20%), survey for NPS |
| 😴 Dormant | Re-engagement campaign, "We miss you" promo, sunset after 2 attempts |